# RAG Pipelines- Data Ingestion to Vector DB Pipeline #

In [3]:
import os
from pathlib import Path

from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Unit 3 BEC 308 Notes.pdf
  ✓ Loaded 15 pages

Total documents loaded: 15


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-06T17:32:40+05:30', 'author': '', 'moddate': '2025-11-06T17:32:40+05:30', 'title': 'Microsoft Word - Unit 1 & 2 Notes', 'source': '..\\data\\text_files\\pdf\\Unit 3 BEC 308 Notes.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Unit 3 BEC 308 Notes.pdf', 'file_type': 'pdf'}, page_content='Unit-III \nAdvanced Machine Learning Techniques and Model Optimization Ensemble \nLearning: Random Forest, Gradient Boosting, and XGBoost. Model \nOptimization Techniques: Hyperparameter tuning (Grid Search, Random \nSearch). Natural Language Processing (NLP): Tokenization, TF-IDF, Word2Vec. \nEthical AI: Bias, fairness, and interpretability in ML models. \nADV ANCED MACHINE LEARNING TECHNIQUES AND \nMODEL OPTIMIZATION ENSEMBLE LEARNING \nWhat is Ensemble Learning? \nIn ensemble learning, instead of training a single model, we train multiple models \nand combine their predictio

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 15 documents into 30 chunks

Example chunk:
Content: Unit-III 
Advanced Machine Learning Techniques and Model Optimization Ensemble 
Learning: Random Forest, Gradient Boosting, and XGBoost. Model 
Optimization Techniques: Hyperparameter tuning (Grid Sea...
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-06T17:32:40+05:30', 'author': '', 'moddate': '2025-11-06T17:32:40+05:30', 'title': 'Microsoft Word - Unit 1 & 2 Notes', 'source': '..\\data\\text_files\\pdf\\Unit 3 BEC 308 Notes.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Unit 3 BEC 308 Notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-06T17:32:40+05:30', 'author': '', 'moddate': '2025-11-06T17:32:40+05:30', 'title': 'Microsoft Word - Unit 1 & 2 Notes', 'source': '..\\data\\text_files\\pdf\\Unit 3 BEC 308 Notes.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Unit 3 BEC 308 Notes.pdf', 'file_type': 'pdf'}, page_content='Unit-III \nAdvanced Machine Learning Techniques and Model Optimization Ensemble \nLearning: Random Forest, Gradient Boosting, and XGBoost. Model \nOptimization Techniques: Hyperparameter tuning (Grid Search, Random \nSearch). Natural Language Processing (NLP): Tokenization, TF-IDF, Word2Vec. \nEthical AI: Bias, fairness, and interpretability in ML models. \nADV ANCED MACHINE LEARNING TECHNIQUES AND \nMODEL OPTIMIZATION ENSEMBLE LEARNING \nWhat is Ensemble Learning? \nIn ensemble learning, instead of training a single model, we train multiple models \nand combine their predictio

### embedding and vector storeDB ###

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'chromadb'

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

NameError: name 'List' is not defined